# XGBoost Baseline — binary classification: fire vs megafire

Input dataset: `data/processed/conaf_enriched_latest.parquet`  
Target variable: `superficie_quemada_total_ha >= MEGAFIRE_HA_THRESHOLD`

**Before running the training cells**, fill in the placeholders in the configuration cell.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PROCESSED = Path("../data/processed")
DATA_MODELS = Path("../data/models")
DATA_MODELS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# ---------------------------------------------------------------
# PLACEHOLDERS -- fill in before training
# ---------------------------------------------------------------
MEGAFIRE_HA_THRESHOLD = None   # TODO: set threshold in hectares (e.g. 500, 1000)

XGB_PARAMS = {
    "n_estimators":     None,   # TODO: number of trees
    "max_depth":        None,   # TODO: maximum depth
    "learning_rate":    None,   # TODO: learning rate (eta)
    "subsample":        None,   # TODO: row fraction per tree (0-1)
    "colsample_bytree": None,   # TODO: column fraction per tree (0-1)
    "scale_pos_weight": None,   # TODO: positive-class weight (compute in cell 4)
}
# ---------------------------------------------------------------

## 1. Load dataset

In [ ]:
df = pd.read_parquet(DATA_PROCESSED / "conaf_enriched_latest.parquet")
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.dtypes.value_counts()

## 2. Target variable exploration

Use this cell to decide the megafire threshold before filling in `MEGAFIRE_HA_THRESHOLD`.

In [ ]:
target_col = "superficie_quemada_total_ha"
s = df[target_col].dropna()

print(s.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
s.clip(upper=s.quantile(0.99)).hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution (without extreme outliers)")
axes[0].set_xlabel("ha")
np.log1p(s).hist(bins=50, ax=axes[1])
axes[1].set_title("log1p distribution")
axes[1].set_xlabel("log(ha + 1)")
plt.tight_layout()
plt.show()

## 3. Target variable definition

In [ ]:
assert MEGAFIRE_HA_THRESHOLD is not None, "Set MEGAFIRE_HA_THRESHOLD in the configuration cell"

df = df.dropna(subset=[target_col]).copy()
df["is_megafire"] = (df[target_col] >= MEGAFIRE_HA_THRESHOLD).astype(int)

balance = df["is_megafire"].value_counts()
print(balance)
print(f"\nRatio negatives/positives (for scale_pos_weight): "
      f"{balance[0] / balance[1]:.2f}")

## 4. Feature selection

In [ ]:
# Columns to exclude from the training set
EXCLUDE = {
    # target and derived
    target_col, "is_megafire",
    # high cardinality / identifiers
    "nombre", "temporada", "datum", "geometry",
    # time columns (derived features extracted below)
    "fecha_hora_inicio", "fecha_inicio", "inicio", "fecha",
    "fecha_hora_inicio_utc", "hora_inicio",
    # ERA5 join internal metrics (potential leakage)
    "era5_dist_km", "era5_dt_hours", "era5_match_quality",
}

# Temporal features derived from the timestamp column
ts_col = next((c for c in ("fecha_hora_inicio", "fecha_inicio", "inicio", "fecha") if c in df.columns), None)
if ts_col:
    ts = pd.to_datetime(df[ts_col], errors="coerce")
    df["month"] = ts.dt.month
    df["hour"] = ts.dt.hour
    df["day_of_year"] = ts.dt.day_of_year

# Ordinal encoding of categorical columns (XGBoost accepts integers)
CAT_COLS = [c for c in ("causa", "alerta", "escenario", "region", "provincia", "comuna") if c in df.columns]
for col in CAT_COLS:
    df[col] = pd.Categorical(df[col]).codes  # -1 for NaN

feature_cols = [
    c for c in df.columns
    if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])
]

X = df[feature_cols]
y = df["is_megafire"]

print(f"Features: {len(feature_cols)}")
print(feature_cols)

## 5. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Positives in train: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Positives in test:  {y_test.sum()} ({y_test.mean():.1%})")

## 6. XGBoost training

In [ ]:
assert all(v is not None for v in XGB_PARAMS.values()), (
    f"Fill in all placeholders in XGB_PARAMS: "
    f"{[k for k, v in XGB_PARAMS.items() if v is None]}"
)

model = xgb.XGBClassifier(
    **XGB_PARAMS,
    eval_metric="auc",
    random_state=RANDOM_STATE,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

## 7. Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["fire", "megafire"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["fire", "megafire"],
    ax=axes[0],
)
axes[0].set_title("Confusion matrix")

# Feature importance (top 20)
importances = pd.Series(model.feature_importances_, index=feature_cols).nlargest(20)
importances.sort_values().plot.barh(ax=axes[1])
axes[1].set_title("Feature importance (top 20)")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

## 8. Save model and metadata

In [ ]:
model_path = DATA_MODELS / "xgboost_baseline.json"
meta_path  = DATA_MODELS / "xgboost_baseline_meta.json"

model.save_model(model_path)

meta = {
    "threshold_ha": MEGAFIRE_HA_THRESHOLD,
    "features": list(X.columns),
    "n_features": len(feature_cols),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "auc_roc": float(auc),
    "xgb_params": XGB_PARAMS,
}
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False))

print(f"Model saved to: {model_path}")
print(f"Metadata saved to: {meta_path}")